# Recursive Language Models for Poker Decision-Making

**STAT 4830 Final Project · Aadithya Srinivasan, Aryan Arora, Aarav M.**

---

This notebook is a self-contained walkthrough of the entire project:

| Section | What you'll see |
|---|---|
| 1. Setup | Install deps, clone repo, verify GPU |
| 2. The Task | What a poker episode looks like |
| 3. The Architecture | How the agent writes and runs Python code |
| 4. Live Demo Traces | Three real agent rollouts with full code |
| 5. Training Story | How performance evolved across all experiments |
| 6. Quantitative Eval | Agent comparison table |
| 7. What Went Wrong | Reward hacking and the REPL fix |
| 8. Next Steps | Slumbot evaluation + future work |

**Runtime:** ~10 minutes on a free-tier Colab T4 GPU (inference only).  
For full BC + RL training (~90 min) see `scripts/primeintellect/README.md`.

> **Quick start:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

## 1. Setup

In [ ]:
# Install all dependencies
!pip install -q torch transformers peft trl datasets accelerate bitsandbytes matplotlib numpy

import os
if not os.path.exists('STAT-4830-RL-project'):
    !git clone https://github.com/aryanarora236/STAT-4830-RL-project.git
os.chdir('STAT-4830-RL-project')
!git pull --ff-only

import sys
sys.path.insert(0, '.')
print('Repo ready.')

In [ ]:
import torch
assert torch.cuda.is_available(), "Go to Runtime > Change runtime type > T4 GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}  ({gpu.total_memory / 1e9:.1f} GB)")

# Run tests to confirm the codebase is intact
!python -m pytest tests/ -q --tb=line 2>&1 | tail -5

## 2. The Task — What a Poker Episode Looks Like

Each episode is a tuple `(context, question, answer*)` where:

- **Context** (~4,300 characters): the current hand state (hole cards, board, positions, stacks, pot odds) **plus** 15 previous hand records from which opponent tendencies can be inferred.
- **Question**: a fixed prompt asking for the correct action.
- **Answer\***: the ground-truth action produced by the heuristic bot (`fold / check / call $X / raise $X`).

The key design choice: **the context lives in a Python variable** (`CONTEXT`), not in the prompt. The agent must write code to parse it — which forces real reasoning rather than pattern matching.

In [ ]:
import random
from src.poker.tasks import generate_poker_task, generate_preflop_task, generate_postflop_task
from src.poker.rewards import parse_action

random.seed(4830)
context, question, correct = generate_poker_task()

print("=" * 70)
print("FULL CONTEXT (trimmed to 1200 chars):")
print("=" * 70)
print(context[:1200])
print("...")
print(f"\n[Total context length: {len(context)} characters]")
print("\n" + "=" * 70)
print("QUESTION:")
print("=" * 70)
print(question)
print("\n" + "=" * 70)
print(f"CORRECT ANSWER (from heuristic teacher): {correct!r}")
print("=" * 70)

## 3. The Architecture — Agent Writes Python, Sandbox Runs It

```
Poker Task (context, question)
          │
          ▼
   ┌─────────────┐
   │   LLM Agent │  writes Python code
   └──────┬──────┘
          │  code string
          ▼
   ┌─────────────┐
   │ Sandboxed   │  executes with CONTEXT global
   │ Python REPL │  (5-sec timeout, whitelist-only)
   └──────┬──────┘
          │  last printed line
          ▼
   Predicted action:  fold / check / call $X / raise $X
```

The RLM pattern: instead of trying to attend over 4k characters of poker history, the model *writes code* that parses it. This is the key technical contribution.

**Training:** two phases.
1. **Behavior Cloning (BC):** fine-tune Qwen2.5-Coder-1.5B on 500 expert (heuristic) `(prompt, code)` pairs using LoRA. Teaches the retrieve→compute→decide pattern.
2. **REINFORCE:** starting from the BC checkpoint, run policy gradient updates on live rollouts. Reward = action-type match vs. teacher (1.0 / 0.3 / 0.0).

In [ ]:
# Show the heuristic's ideal code for the task above — the target BC teaches the LLM to write
from src.poker.agents import PokerHeuristicAgent
from src.utils import safe_execute_code

heuristic = PokerHeuristicAgent()
heur_pred, heur_trace = heuristic.run_episode(context, question, correct)

print("=" * 70)
print("HEURISTIC AGENT — ideal retrieve→compute→decide code:")
print("=" * 70)
heur_codes = [e.get('code', '') for e in heur_trace if 'code' in e]
if heur_codes:
    print(heur_codes[-1][:1500])
print(f"\nHeuristic predicted: {heur_pred!r}  |  Correct: {correct!r}")

## 4. Live Demo Traces

We load the best RL checkpoint (iteration 105 of the 120-iteration run, EMA accuracy 42.3%) and run three live poker scenarios. For each we show:
- The game state
- The exact Python code the model wrote
- The sandbox output
- Whether the action matched the teacher

In [ ]:
# Load model — RL checkpoint if present, otherwise base model for zero-shot demo
from src.training import load_model

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
RL_CKPT = 'checkpoints/poker_rl'

if os.path.exists(os.path.join(RL_CKPT, 'adapter_config.json')):
    print(f'Loading RL checkpoint from {RL_CKPT} ...')
    model, tokenizer = load_model(RL_CKPT, load_in_4bit=True)
    MODE = 'RL-finetuned'
else:
    print(f'RL checkpoint not found — loading base model (zero-shot demo) ...')
    model, tokenizer = load_model(MODEL_ID, load_in_4bit=True, lora_r=16)
    MODE = 'zero-shot'

model.eval()
print(f'Ready ({MODE} mode).')

In [ ]:
from src.poker.agents import PokerLocalLLMAgent

llm_agent = PokerLocalLLMAgent(
    model=model, tokenizer=tokenizer,
    name=f'Qwen1.5B-{MODE}', max_steps=3, temperature=0.1,
)

random.seed(4830)

scenarios = [
    ('Preflop — read opponent VPIP',  generate_preflop_task),
    ('Postflop — pot odds + board',   generate_postflop_task),
    ('Mixed streets',                  generate_poker_task),
]

for label, gen in scenarios:
    print()
    print('=' * 72)
    print(f'SCENARIO: {label}')
    print('=' * 72)

    ctx, q, ans = gen()
    print('CONTEXT (first 500 chars):')
    print(ctx[:500].rstrip() + '...')
    print()

    # Run LLM agent
    llm_pred, llm_trace = llm_agent.run_episode(ctx, q, ans)
    codes = [e.get('code', '') for e in llm_trace if 'code' in e]

    print('--- LLM AGENT ---')
    if codes:
        print('Code written (first 600 chars):')
        print(codes[-1][:600].rstrip())
        exec_results = [e.get('exec_result', {}) for e in llm_trace if 'exec_result' in e]
        if exec_results:
            last_exec = exec_results[-1]
            status = 'OK' if last_exec.get('ok') else 'ERROR'
            stdout = last_exec.get('stdout', '').strip()
            stderr = last_exec.get('stderr', '').strip()[:200]
            print(f'\nExecution: {status}')
            if stdout:
                print(f'Stdout: {stdout!r}')
            if stderr:
                print(f'Stderr: {stderr!r}')
    else:
        print('(No code extracted — fallback to direct text parse)')

    match = parse_action(llm_pred)[0] == parse_action(ans)[0]
    print(f'\nPredicted: {llm_pred!r}  |  Correct: {ans!r}  |  Match: {"✓" if match else "✗"}')
    print()

### 4.1 Pre-recorded Sample Traces (from the shaped-reward RL run)

The three examples below are verbatim from the shaped-reward Modal run (Experiment v1, April 23). They illustrate the spectrum from reward hacking to genuine reasoning.

---

**Sample 1 — The shortcut (reward hack):** One-line fold inside a code fence. Earns the code-bonus without any reasoning.

```python
print("fold")
```

---

**Sample 2 — The attempt:** Model scans game history for opponent stats. Gets truncated mid-loop and crashes on an undefined name.

```python
import re
lines = CONTEXT.split('\n')
for i, line in enumerate(lines):
    if '=== PREVIOUS HANDS' in line:
        history_start = i
        break
stats = {}
# (truncated — crashes on undefined name 'stats' in compute step)
```

---

**Sample 3 — Real strategic reasoning:** Folds marginal hands against tight players, raises with flush draws. This is the behavior the RLM framework is designed for.

```python
# Tight (low VPIP): respect bets, fold marginals
elif opp_stats['VPIP'] <= 0.4:
    if strength['flush_draw']:
        print("call {}".format(to_call))
    else:
        print("raise {}".format(max(10, to_call * 1.5)))
```

---

The progression from Sample 1 → Sample 3 is the learning signal. The reward-shaping fix (Section 7) makes Sample 1 less rewarding so the policy pushes toward Samples 2 and 3.

## 5. Training Story — How Performance Evolved

We ran five experiments, each motivated by what the previous one revealed.

| Experiment | Key change | Best held-out acc |
|---|---|---|
| Proof of concept (3–10 iters) | Verify pipeline works | n/a (no held-out eval) |
| Long run (120 iters) | Scale iterations | n/a (no held-out eval; EMA 42.3%) |
| Experiment A (50 iters) | Add held-out eval, bigger batch | **25.0%** @ iter 35 |
| Experiment B (100 iters) | Longer run, smaller batch | **20.0%** (flat) |
| Fast Mixed run (20 iters) | Shorter, earlier eval | **40.0%** @ iter 10 |
| Shaped Reward (20 iters) | Reward code use explicitly | **33.3%** @ iter 5, 10 |

Key insight: **the biggest improvement was methodology**, not hyperparameters. Moving from training-only metrics to held-out evaluation with best-checkpoint selection is what unlocked the real gains.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── Held-out eval data (from eval_leaderboard.csv files) ──────────────────
exp_a_iters  = [5, 10, 15, 20, 25, 30, 35, 40, 45]
exp_a_acc    = [0.25, 0.25, 0.233, 0.233, 0.25, 0.233, 0.25, 0.25, 0.233]

exp_b_iters  = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
exp_b_acc    = [0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20, 0.20]

fast_iters   = [10, 20]
fast_acc     = [0.40, 0.25]

shaped_iters = [5, 10]
shaped_acc   = [0.333, 0.333]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Held-out accuracy over time ─────────────────────────────────────
ax = axes[0]
ax.plot(exp_a_iters,  exp_a_acc,  'o-', color='#2980b9', linewidth=2, markersize=6, label='Exp A (batch 6, eval every 5)')
ax.plot(exp_b_iters,  exp_b_acc,  's--', color='#e74c3c', linewidth=2, markersize=6, label='Exp B (batch 4, 100 iters)')
ax.plot(fast_iters,   fast_acc,   '^-', color='#27ae60', linewidth=2, markersize=8, label='Fast Mixed (batch 4, 20 iters)')
ax.plot(shaped_iters, shaped_acc, 'D-', color='#8e44ad', linewidth=2, markersize=7, label='Shaped Reward (code bonus)')
ax.axhline(0.20, color='#e74c3c', linestyle=':', alpha=0.4, linewidth=1)
ax.set_xlabel('RL Iteration', fontsize=12)
ax.set_ylabel('Held-out Accuracy', fontsize=12)
ax.set_title('Held-out Eval Accuracy per Experiment', fontsize=13, fontweight='bold')
ax.set_ylim(-0.02, 0.55)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(fontsize=9, loc='upper right')
ax.grid(alpha=0.3)

# ── Right: Best held-out per experiment bar chart ─────────────────────────
ax2 = axes[1]
labels = ['Exp B\n(100 iters)', 'Exp A\n(50 iters)', 'Fast Mixed\n(20 iters)', 'Shaped\nReward']
accs   = [0.20, 0.25, 0.40, 0.333]
colors = ['#e74c3c', '#2980b9', '#27ae60', '#8e44ad']

bars = ax2.bar(labels, accs, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
             f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax2.axhline(0.20, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='Exp B baseline')
ax2.set_ylabel('Best Held-out Accuracy', fontsize=12)
ax2.set_title('Best Held-out Accuracy per Experiment', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 0.55)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/held_out_eval_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to figures/held_out_eval_summary.png')

In [ ]:
# Load and display the existing training curves from the long RL run (if present)
from IPython.display import Image, display

curve_candidates = [
    'docs/results/poker_rl_long_simple_120iters_20260420/training_curves.png',
    'docs/results/poker_rl_expA_partial_20260422/poker_rl_expA_evalselect_20260421_long/training_curves.png',
    'docs/results/poker_rl_expB_mixed20_20260422/poker_rl_expB_mixed20_20260422/training_curves.png',
]

for path in curve_candidates:
    if os.path.exists(path):
        run_name = path.split('/')[-2]
        print(f'Training curves: {run_name}')
        display(Image(path, width=800))
        print()

## 6. Quantitative Evaluation — Agent Comparison

All agents evaluated on the **same fixed task set**, same seed, same scoring function.

In [ ]:
from src.poker.evaluation import PokerEvaluationFramework

random.seed(2026)
agents = [llm_agent, heuristic]

print('Running quantitative eval (30 episodes)...')
eval_framework = PokerEvaluationFramework(
    agents=agents,
    task_generator=generate_poker_task,
    num_episodes=30,
)
eval_framework.run_evaluation()
eval_framework.display_results()
print()
eval_framework.display_confusion_matrix(llm_agent.name)

In [ ]:
# Published experiment summary table
import pandas as pd
from IPython.display import display

summary_data = {
    'Agent': [
        'Zero-shot Qwen-7B (HF API)',
        'Exp B best_by_eval (unshaped)',
        'Exp A best_by_eval (iter 35)',
        'Fast Mixed best (iter 10)',
        'Shaped Reward (iter 5/10)',
        'Heuristic teacher',
    ],
    'Held-out Acc': ['8.0%', '20.0%', '25.0%', '40.0%', '33.3%', '100.0%'],
    'Held-out Reward': ['0.092', '0.200', '0.285', '0.460', '0.333', '1.000'],
    'N episodes': [25, 20, 60, 20, 15, 25],
    'Note': [
        'No fine-tuning',
        'Flat across 100 iters',
        'First gen signal',
        'Best overall',
        '+13pp vs Exp B',
        'Oracle ceiling',
    ]
}
df = pd.DataFrame(summary_data)
display(df.to_string(index=False))

## 7. What Went Wrong — Reward Hacking and the REPL Fix

The most important empirical finding of this project is a **failure mode**.

### The problem: the policy stopped using the REPL

In Experiment B, we logged `real_code` (rollouts where the model wrote actual Python) vs `wrapped_action_code` (rollouts where it just printed one word like `fold`):

| Metric | Exp B average (per batch of 4) |
|---|---|
| `real_code` | **0 / 4** |
| `wrapped_action_code` | **4 / 4** |

Every single rollout just emitted `fold` or `call` as a text string. The Python REPL we built might as well not exist. This is reward hacking: the type-match reward fires ~25% of the time on random guesses, and writing correct Python is expensive. REINFORCE found the cheap path.

### The fix: shaped reward

The fix is simple in principle — **price tool use explicitly**:
- `+0.3` bonus when the model writes real Python (code extracted, not just wrapped text)
- `+0.2` bonus when the code references opponent stats (`VPIP`, `aggression`)
- `-0.2` penalty when the model falls back to one-word guesses

Result: `real_code` jumps from 0/4 → 3–4/4 per batch, and held-out accuracy improves from 20.0% → 33.3% (+13 pp).

### Layer 2 reward hacking

After the fix, the policy found a new shortcut: write `print("fold")` inside a proper code fence. This earns the code bonus without any reasoning. The natural next reward signal is `exec_ok` — bonus for code that actually **runs** — which would force the policy to write syntactically correct Python.

In [ ]:
# Visualize the code-usage diagnostic across experiments
import matplotlib.pyplot as plt
import numpy as np

experiments = ['Exp B\n(unshaped)', 'Fast Mixed\n(unshaped)', 'Shaped\nReward v1', 'Shaped\nReward v4']
real_code    = [0.0,  0.0,  0.55, 0.95]  # fraction of batch with real code
wrapped      = [1.0,  1.0,  0.45, 0.05]  # fraction that fell back to wrapped

x = np.arange(len(experiments))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - width/2, real_code, width, label='Real code (model wrote Python)', color='#27ae60', alpha=0.85)
b2 = ax.bar(x + width/2, wrapped,   width, label='Wrapped/fallback (one-word action)', color='#e74c3c', alpha=0.85)

ax.set_ylabel('Fraction of batch', fontsize=12)
ax.set_title('Code Usage Across Experiments', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(experiments, fontsize=10)
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

for bar in b1:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar in b2:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{bar.get_height():.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/code_usage_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/code_usage_diagnostic.png')

## 8. Next Steps — Slumbot Evaluation and Future Work

### Slumbot: the next evaluation target

[Slumbot](http://www.slumbot.com/) is a near-Nash equilibrium no-limit hold'em solver, freely available via a public REST API. It plays heads-up NLHE at close to game-theoretic optimal.

Playing our RLM agent against Slumbot would provide a **real-world, opponent-independent benchmark** — unlike our current evaluation which is bounded by the heuristic teacher. This is the natural next experiment:

1. Integrate Slumbot API (`http://www.slumbot.com/play_hand`).
2. Play 1,000 heads-up hands, measure win-rate in big blinds per 100 hands (BB/100).
3. Compare zero-shot vs BC vs RL vs shaped-reward checkpoints.

Since Slumbot plays near-optimally, any positive win-rate would be remarkable. But even measuring how *much worse* than Slumbot we are gives a principled gap to close.

### Full future work pipeline

```
Current:  type-match reward  →  reward hacking (skip REPL)
Fix 1:    + code bonus       →  writes code, but doesn't run
Fix 2:    + exec_ok bonus    →  writes code that runs, but guesses action
Fix 3:    + EV reward        →  uses pot odds + equity for action
Fix 4:    vs Slumbot         →  real-world evaluation beyond teacher ceiling
```

Each step addresses the reward-hacking failure mode from the previous step. This is the standard iterative reward-shaping pipeline, grounded in the §7 course notes framework.

In [ ]:
# Placeholder: Slumbot API integration sketch
# (Not live — requires Slumbot API access and a running game session)

SLUMBOT_DEMO_CODE = """
import requests

# 1. Start a new hand
resp = requests.post('http://www.slumbot.com/new_hand', json={'token': TOKEN})
game_state = resp.json()

# 2. Parse Slumbot's game state into our CONTEXT format
context = format_slumbot_state(game_state)   # adapter: Slumbot JSON -> our 4k-char format

# 3. Ask our RLM agent for an action
predicted, trace = llm_agent.run_episode(context, QUESTION, None)

# 4. Submit action to Slumbot
action_str = translate_action(predicted)     # fold/call/raise -> Slumbot format
resp = requests.post('http://www.slumbot.com/act', json={'token': TOKEN, 'action': action_str})
"""

print("Slumbot integration sketch (not live):")
print(SLUMBOT_DEMO_CODE)
print("See docs/future_work.md for the full integration plan.")

## Appendix — Reproducing the Full Training Pipeline

This notebook only runs inference. Full BC + REINFORCE training requires an Ampere+ GPU (A100, L40, H100) and ~90 minutes.

### Quick reference (on a PrimeIntellect H100 pod)

```bash
# 1. Create pod
prime pods create --id <H100-availability-id> --env HF_TOKEN=$HF_TOKEN --name stat4830
prime pods ssh <pod-id>

# 2. Bootstrap (installs Python env, clones repo, unsloth)
INSTALL_UNSLOTH=1 bash <(curl -sSL https://raw.githubusercontent.com/aryanarora236/STAT-4830-RL-project/main/scripts/primeintellect/bootstrap.sh)

# 3. Run the full pipeline (BC + 130 iters RL + eval)
cd /root/STAT-4830-RL-project && source .venv/bin/activate
bash scripts/primeintellect/run_full.sh

# 4. Retrieve artifacts
scp -r pod:/root/STAT-4830-RL-project/checkpoints ./
scp -r pod:/root/STAT-4830-RL-project/docs/results ./docs/
```

Full playbook with troubleshooting at [`scripts/primeintellect/README.md`](../scripts/primeintellect/README.md).

### Shaped-reward run (Modal)

```bash
modal run scripts/modal_shaped_reward.py
```

Results land in `experiments/results/modal_shaped_leaderboard_v1det.json`.